# Concrete Strength Prediction

Predict concrete compressive strength from eight mixture and age features.

See [README.md](README.md) for data, setup, workflow, and evaluation notes.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim

In [ ]:
concrete_df = pd.read_excel("concrete_data.xls")

In [ ]:
concrete_df.head()

In [ ]:
concrete_df.info()

In [ ]:
concrete_df.describe()

In [ ]:
concrete_df.isnull().sum()

In [ ]:
concrete_df.columns

In [ ]:
X = concrete_df.drop("Concrete compressive strength(MPa, megapascals) ",axis=1)
y = concrete_df["Concrete compressive strength(MPa, megapascals) "]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(

            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            nn.Linear(6,6),
            nn.ReLU(),

            nn.Linear(6,1),
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
train_losses = []
val_losses = []
epochs = 100

best_val_loss = float("inf") 

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    for xb,yb in train_loader:
    
        optimizer.zero_grad()
    
        outputs = model(xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    
        running_loss += loss.item()
        
    epoch_train_loss = running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs,yb)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)

    print(
    f"Epoch {epoch+1}/{epochs} ==> "
    f"Training Loss = {epoch_train_loss:.4f} | "
    f"Validation Loss = {epoch_val_loss:.4f}"
)       

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt") #pt or pth file

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss" : train_losses,
    "Validation Loss" : val_losses
})

plt.figure(figsize=(10,8))
plt.plot(loss_df["Training Loss"],label="Training Loss")
plt.plot(loss_df["Validation Loss"],label="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
#Evaluate our Model

model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criterion(train_preds,y_train_tensor)
    test_mse_loss = criterion(test_preds,y_test_tensor)
    
print("Training MSE : ",train_mse_loss.item())
print("Testing MSE : ",test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("R2 Score : ",r2_score(y_test,test_preds)*100)

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(),columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values,columns=["Actual Values"])

pd.concat([predicted_df,actual_df],axis=1)